# Agentic NAS Workflow: Step-by-Step Exploration
Welcome to the interactive exploration of our Nextcloud ingestion and deduplication pipeline! In this notebook, we will walk through the core logic powering our automated NAS workflow.

In [1]:
# Setup: Import our custom modules from the `src` package
import os
import sys

# Ensure the project root is in the Python path
sys.path.append(os.path.abspath('..'))

from src.config import INGEST_DIR
from src.dedupe.crypto import calculate_blake3
# from src.storage.webdav import upload_file

print("Modules loaded successfully!")
print(f"Target Ingest Directory: {INGEST_DIR}")

Modules loaded successfully!
Target Ingest Directory: /cloud_ingest/gdrive_ahabib9387


## Step 1: Data Ingestion and Mock File Generation
First, let's explore our local ingestion directory. This is where files land before they are processed by the pipeline. We will create some mock files (including an exact duplicate) to demonstrate our pipeline's capabilities.

In [2]:
DUMMY_INGEST_DIR = "/home/coder/projects/agentic-nas-workflow/notebooks/cloud_ingest"

In [ ]:
# Let's see what files we have in the ingest directory
if not os.path.exists(DUMMY_INGEST_DIR):
    print(f"Directory {DUMMY_INGEST_DIR} does not exist yet. Let's create some dummy files for this demo.")
    os.makedirs(DUMMY_INGEST_DIR, exist_ok=True)
    
    with open(os.path.join(DUMMY_INGEST_DIR, "report.pdf"), "w") as f:
        f.write("Dummy PDF content")
    with open(os.path.join(DUMMY_INGEST_DIR, "photo.jpg"), "w") as f:
        f.write("Dummy Image content")
    # A duplicate file
    with open(os.path.join(DUMMY_INGEST_DIR, "report_copy.pdf"), "w") as f:
        f.write("Dummy PDF content")

files_to_process = []
for root, _, files in os.walk(DUMMY_INGEST_DIR):
    for file in files:
        files_to_process.append(os.path.join(root, file))

print(f"Found {len(files_to_process)} files to process:")
for f in files_to_process:
    print(f" - {f}")

Directory ./cloud_ingest does not exist yet. Let's create some dummy files for this demo.
Found 3 files to process:
 - ./cloud_ingest/report_copy.pdf
 - ./cloud_ingest/report.pdf
 - ./cloud_ingest/photo.jpg


## Step 2: Cryptographic Deduplication (BLAKE3 Hashing)
To prevent duplicate files from eating up precious NAS/Nextcloud space, we use **BLAKE3 Hashing** for exact bitwise deduplication.
Unlike md5 or sha256, BLAKE3 is extremely fast and can stream large files efficiently in chunks.

Let's compute the hashes for our three generated files and verify that our duplicate detector successfully flags the matching PDF files while letting the unique image pass.

In [4]:
# Compute and compare BLAKE3 hashes
pdf_path = os.path.join(DUMMY_INGEST_DIR, "report.pdf")
copy_path = os.path.join(DUMMY_INGEST_DIR, "report_copy.pdf")
photo_path = os.path.join(DUMMY_INGEST_DIR, "photo.jpg")

hash_pdf = calculate_blake3(pdf_path)
hash_copy = calculate_blake3(copy_path)
hash_photo = calculate_blake3(photo_path)

print(f"report.pdf hash:       {hash_pdf}")
print(f"report_copy.pdf hash:  {hash_copy}")
print(f"photo.jpg hash:        {hash_photo}\n")

# Assert and show results
print(f"Does report.pdf match report_copy.pdf? {hash_pdf == hash_copy} (Expected: True)")
print(f"Does report.pdf match photo.jpg?        {hash_pdf == hash_photo} (Expected: False)")

assert hash_pdf == hash_copy, "Error: PDF copy should have the same hash!"
assert hash_pdf != hash_photo, "Error: Image and PDF should have different hashes!"
print("\nSuccess! Cryptographic deduplication is working perfectly.")

report.pdf hash:       f9ea87c1bcce628cc6260f8d832dabe258857fd2513980bccf263554db65d5fe
report_copy.pdf hash:  f9ea87c1bcce628cc6260f8d832dabe258857fd2513980bccf263554db65d5fe
photo.jpg hash:        53f34198c1152cadab64c5745d9ecff100a9f1cf0aa9610d2cbebdd01ba98bcf

Does report.pdf match report_copy.pdf? True (Expected: True)
Does report.pdf match photo.jpg?        False (Expected: False)

Success! Cryptographic deduplication is working perfectly.


## Step 3: Semantic Deduplication (Content Similarity)
Cryptographic deduplication is perfect for finding **exact, byte-for-byte identical files**. However, it fails if a file is slightly modified (e.g., a PDF report with a single updated typo or a draft and a final version of a text file).

This is where **Semantic Deduplication** comes in. By analyzing the textual or visual content of files, we can calculate a **similarity score** and identify files that are nearly identical.

In `src/dedupe/semantic.py`, we implemented:
1. **Jaccard Similarity**: Measuring the overlap of unique words between documents.
2. **Cosine Similarity**: Comparing term frequency vectors to measure semantic alignment.

Let's test this semantic similarity logic on three sample documents!

In [7]:
# Import and test semantic deduplication
from src.dedupe.semantic import calculate_jaccard_similarity, calculate_cosine_similarity

# Let's define three texts to compare
doc_original = "The agentic NAS workflow automates our file ingestion, cryptographic hashing, and remote uploads to Nextcloud."
doc_near_duplicate = "The agentic NAS workflow automatically handles our file ingestion, cryptographic hashing, and remote uploads to Nextcloud server."
doc_completely_different = "Tomorrow's weather forecast predicts clear skies with a mild breeze and temperatures around 72 degrees."

print("Document Comparisons:")
print("-" * 50)

# Compare original vs near duplicate
jaccard_near = calculate_jaccard_similarity(doc_original, doc_near_duplicate)
cosine_near = calculate_cosine_similarity(doc_original, doc_near_duplicate)
print("Original vs Near Duplicate:")
print(f"  - Jaccard Similarity: {jaccard_near:.4f}")
print(f"  - Cosine Similarity:  {cosine_near:.4f}")

# Compare original vs different
jaccard_diff = calculate_jaccard_similarity(doc_original, doc_completely_different)
cosine_diff = calculate_cosine_similarity(doc_original, doc_completely_different)
print("\nOriginal vs Completely Different:")
print(f"  - Jaccard Similarity: {jaccard_diff:.4f}")
print(f"  - Cosine Similarity:  {cosine_diff:.4f}")

# Assert similarity behaves as expected
assert cosine_near > 0.80, "Error: Near duplicates should have high similarity!"
assert cosine_diff < 0.15, "Error: Completely different documents should have low similarity!"
print("\nSuccess! Semantic deduplication detects near-duplicates perfectly.")

Document Comparisons:
--------------------------------------------------
Original vs Near Duplicate:
  - Jaccard Similarity: 0.7778
  - Cosine Similarity:  0.8767

Original vs Completely Different:
  - Jaccard Similarity: 0.0333
  - Cosine Similarity:  0.0645

Success! Semantic deduplication detects near-duplicates perfectly.


## Step 4: WebDAV Storage and Prefect Orchestration
Once our files are ingested and checked for duplicates, they are routed and uploaded idempotently to **Nextcloud** using WebDAV (`webdav4`).

The entire process is orchestrated as a pipeline flow using **Prefect 3.0**, with task retries and exponential backoffs configured to handle network hiccups seamlessly.

Let's check out our WebDAV upload helper and run the complete pipeline!
*(Note: To make this exploration safe and robust, if your Nextcloud instance or Prefect server is not currently reachable, the cells will catch the connection exceptions and describe the simulated production behavior instead of crashing!)*

In [3]:
# Test uploading a single file to Nextcloud WebDAV
from src.config import NEXTCLOUD_URL, INGEST_DIR
from prefect.blocks.system import Secret
from webdav4.client import Client

In [4]:
print(f"Targeting Nextcloud at: {NEXTCLOUD_URL}")
print(f"Scanning Ingest Directory: {INGEST_DIR}")

Targeting Nextcloud at: http://192.168.1.55:30027/remote.php/webdav
Scanning Ingest Directory: /cloud_ingest/gdrive_ahabib9387


In [5]:
# 1. Fetch secrets safely inside Jupyter's async loop
user_block = await Secret.load("nextcloud-username")
pass_block = await Secret.load("nextcloud-password")

# 2. Instantiate the WebDAV client (This is our Dependency!)
client = Client(NEXTCLOUD_URL, auth=(user_block.get(), pass_block.get()))

print("\nWebDAV Client successfully instantiated.")
print("Root Nextcloud Folders:")
for item in client.ls("/"):
    print(f" - {item['name']}")


WebDAV Client successfully instantiated.
Root Nextcloud Folders:
 - Photos
 - Documents
 - Templates


In [8]:
DUMMY_INGEST_DIR

'/home/coder/projects/agentic-nas-workflow/notebooks/cloud_ingest'

In [6]:
remote_test_dir = "/Documents/IngestionTest"

# Idempotently create the remote folder if it doesn't exist
if not client.exists(remote_test_dir):
    client.mkdir(remote_test_dir)
    print(f"Created remote folder: {remote_test_dir}")

if not os.path.exists(DUMMY_INGEST_DIR):
    print(f"CRITICAL ERROR: Could not find local folder at {DUMMY_INGEST_DIR}")
else:
    print(f"Scanning local directory: {DUMMY_INGEST_DIR}...\n")
    
    for filename in os.listdir(DUMMY_INGEST_DIR):
        local_path = os.path.join(DUMMY_INGEST_DIR, filename)
        
        # SRE Check: Ensure we are only uploading files, not sub-directories
        if os.path.isfile(local_path):
            remote_path = f"{remote_test_dir}/{filename}"
            print(f" ➔ Uploading: {filename}...")
            
            # Using upload_file with from_path and to_path
            client.upload_file(from_path=local_path, to_path=remote_path, overwrite=True)
            print(f"    [Success] Uploaded to {remote_path}")

# Verify the State Database (Nextcloud)
print(f"\nVerification - Files currently in Nextcloud {remote_test_dir}:")
for item in client.ls(remote_test_dir):
    # We filter out the directory itself and just show the files
    if item['type'] == 'file':
        # Convert bytes to MB for easier reading
        size_mb = item.get('size', 0) / (1024 * 1024)
        print(f" - {item['name']} ({size_mb:.2f} MB)")

Created remote folder: /Documents/IngestionTest
Scanning local directory: /home/coder/projects/agentic-nas-workflow/notebooks/cloud_ingest...

 ➔ Uploading: report_copy.pdf...
    [Success] Uploaded to /Documents/IngestionTest/report_copy.pdf
 ➔ Uploading: report.pdf...
    [Success] Uploaded to /Documents/IngestionTest/report.pdf
 ➔ Uploading: photo.jpg...
    [Success] Uploaded to /Documents/IngestionTest/photo.jpg

Verification - Files currently in Nextcloud /Documents/IngestionTest:
 - Documents/IngestionTest/report_copy.pdf (0.00 MB)
 - Documents/IngestionTest/report.pdf (0.00 MB)
 - Documents/IngestionTest/photo.jpg (0.00 MB)


In [2]:
# Run the complete SRE ingestion and organization pipeline
from src.main import run_pipeline

In [ ]:
try:
    print("Initializing Agentic NAS Pipeline flow...")
    run_pipeline()
except Exception as e:
    print("\n[Environment Notice]: Flow execution completed with exception/warning.")
    print(f"Error detail: {e}")
    print("This is normal when Prefect or Nextcloud services are not fully running locally.")

Initializing Agentic NAS Pipeline flow...


12:28:22.865 | INFO    | Flow run 'huge-zebu' - Beginning flow run 'huge-zebu' for flow 'Agentic-NAS-Pipeline'

12:28:22.883 | INFO    | Flow run 'huge-zebu' - View at http://192.168.1.55:4200/runs/flow-run/19fc730a-02b2-41d5-b219-ebfee511a75e

12:28:22.889 | INFO    | Flow run 'huge-zebu' - Starting SRE Pipeline on directory: /cloud_ingest/gdrive_ahabib9387

12:28:22.896 | INFO    | Flow run 'huge-zebu' - Authenticating with Prefect Vault...

...

12:29:49.637 | INFO    | Flow run 'huge-zebu' - FAILED to upload Line.Of.Duty.S06E03.720p.AMZN.WEBRip.x264-GalaxyTV.mkv: 'Client' object has no attribute 'upload_sync'

12:29:49.643 | INFO    | Flow run 'huge-zebu' - 
Processing: Line.Of.Duty.S06E04.720p.AMZN.WEBRip.x264-GalaxyTV.mkv

12:29:50.824 | INFO    | Task run 'task_hash_file-fa9' - Finished in state Completed()

12:29:50.839 | ERROR   | Task run 'task_upload_file-981' - Error encountered when computing cache key - result will not be persisted.
Traceback (most recent call last):
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/prefect/cache_policies.py", line 386, in compute_key
    return hash_objects(hashed_inputs, raise_on_failure=True)
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/prefect/utilities/hashing.py", line 89, in hash_objects
    raise HashError(msg)
prefect.exceptions.HashError: Unable to create hash - objects could not be serialized.
  JSON error: Unable to serialize unknown type: <class 'webdav4.client.Client'>
  Pickle error: cannot pickle '_thread.RLock' object

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/prefect/task_engine.py", line 285, in compute_transaction_key
    key = self.task.cache_policy.compute_key(
        task_ctx=task_run_context,
        inputs=self.parameters or {},
        flow_parameters=parameters or {},
    )
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/prefect/cache_policies.py", line 218, in compute_key
    policy_key = policy.compute_key(
        task_ctx=task_ctx,
    ...<2 lines>...
        **kwargs,
    )
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/prefect/cache_policies.py", line 396, in compute_key
    raise ValueError(msg) from exc
ValueError: Unable to create hash - objects could not be serialized.
  JSON error: Unable to serialize unknown type: <class 'webdav4.client.Client'>
  Pickle error: cannot pickle '_thread.RLock' object

This often occurs when task inputs contain objects that cannot be cached like locks, file handles, or other system resources.

To resolve this, you can:
  1. Exclude these arguments by defining a custom `cache_key_fn`
  2. Disable caching by passing `cache_policy=NO_CACHE`

12:29:51.014 | INFO    | Task run 'task_upload_file-981' - Task run failed with exception: AttributeError("'Client' object has no attribute 'upload_sync'") - Retry 1/3 will start 5 second(s) from now